In [1]:
!pip uninstall -y torchao
!pip install -q "diffusers>=0.27.0" transformers accelerate safetensors peft

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
"""
stage9_testset_map_evaluation.py
============================================================================
APPROACH A — evaluate uncertainty maps on the HELD-OUT TEST SPLIT.

WHY THIS RUN EXISTS
Stage 4 compared diffusion-ensemble variance against MC-Dropout variance and
MCD won nearly everywhere. That comparison was not like-for-like: the
DenseNet supplying the MCD maps had been trained supervised on those exact
images, while RoentGen was zero-shot. On the test split neither model has
seen the images, so the comparison becomes fair. That is the whole point of
Approach A -- it does not add a new method, it removes a confound.

UNIT OF ANALYSIS -- READ THIS BEFORE CHANGING ANYTHING
One AUROC per image, then bootstrap over IMAGES. Pooling every pixel from
every image into one ROC would give ~1.7e8 "samples", but pixels within an
image are heavily correlated and large-lesion images would dominate. The
resulting CI would be absurdly narrow and wrong. If the Stage 1 train-set
numbers were computed by pooling, recompute them this way before quoting
train and test side by side.

THREE METHODS, AND WHAT EACH ONE ACTUALLY IS
  diffusion     pixel-wise variance across N=20 stochastic reconstructions
                from a fixed inverted latent. Zero-shot: RoentGen never saw
                these images and never saw the labels.
  mcd_logit     MC-Dropout predictive variance from the DenseNet. Zero-shot
                w.r.t. these IMAGES on the test split, but the model was
                trained on this LABEL SET. Epistemic uncertainty.
  mcd_gradcam   Grad-CAM saliency from the same DenseNet.

  READ THE GRAD-CAM ROW WITH CARE. It is NOT an uncertainty estimate. It is
  a class-evidence attribution: it answers "which pixels drove the positive
  prediction", which is almost the definition of the localisation task being
  scored. Diffusion variance answers "where is the generative model
  unsure", which only coincides with the lesion if instability tracks
  pathology. Grad-CAM is therefore a SKYLINE -- an upper reference for what
  a label-supervised method achieves -- not a peer baseline. Reporting it as
  though diffusion "lost" to it misstates the comparison.

  Two further asymmetries to state in the thesis:
    - Both MCD variants come from a classifier trained with the very labels
      the boxes encode. The test split removes the image-level confound that
      broke Stage 4; it does NOT remove label supervision.
    - Grad-CAM maps originate at the last conv layer (~16x16) and are
      upsampled to 512. Large smooth blobs score well on box-overlap metrics
      almost mechanically. Compare the effective resolution of the three map
      types before attributing any gap to method quality.

METRICS, and why each is here
  pixel_auroc   ranking metric: does variance rank in-box pixels above
                out-of-box pixels? Threshold-free, prevalence-independent.
  pixel_auprc   paired with the in-box pixel fraction as the baseline. Boxes
                cover a small share of the image, so AUPRC must be read as
                lift over that fraction, never against 0.5.
  dice_at_p     Dice after binarising the map at its own P-th percentile.
                Threshold chosen from the MAP, not from the labels, so no
                leakage. Reported across several P since Dice is threshold-
                sensitive in a way AUROC is not.
  pointing_hit  does the single highest-variance pixel fall inside a box?
                Crude but robust to map scaling and to calibration, and it
                is the metric a clinician's intuition actually matches.

STATISTICS
  - Bootstrap over images (B=2000) for the CI on the mean.
  - Wilcoxon signed-rank against the 0.5 chance line for AUROC. Non-
    parametric because per-image AUROC is bounded and skewed.
  - Friedman test across all three methods on the images ALL of them cover.
    With three related samples, running three pairwise Wilcoxons alone
    inflates the error rate; Friedman is the omnibus that licenses the
    post-hoc pairs.
  - Post-hoc paired Wilcoxon on every method pair, BH-corrected across the
    whole pair x class x metric family.

CONTAMINATION LOGIC IS INVERTED HERE
Stage 7 asserts NO test image appears in any generated map. This script
asserts the opposite for its own inputs: these maps MUST be test images.
It also re-asserts that these maps were never used in Stage 6 training.
Both checks run; either failing invalidates the run.
"""

import json
import warnings
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import zoom
from sklearn.metrics import roc_auc_score, average_precision_score
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

# =============================================================================
# CONFIG — verify every path against the Kaggle sidebar before running
# =============================================================================

# One entry per class. Confirmed against the actual inspect_maps() output --
# every one of these directories was verified to contain chunked .npz files
# keyed "{image_id}__mean" / "{image_id}__variance" (diffusion files also
# carry "{image_id}__z_T", ignored via CHUNK_IGNORE_SUFFIXES above).
DIFFUSION_ROOT = "/kaggle/input/datasets/kartikichandratre"
MCD_LOGIT_ROOT = ("/kaggle/input/datasets/kartikichandratre/"
                  "stage4-mc-dropout-logit-testbaseline/stage4_mc_dropout_maps")
MCD_GRADCAM_ROOT = ("/kaggle/input/datasets/kartikichandratre/"
                    "stage4-mc-dropout-gradcam-testbaseline/stage4_mc_dropout_maps")

TEST_MAP_DIRS = {
    "Pneumothorax": {
        "diffusion":   f"{DIFFUSION_ROOT}/pneumothorax-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/pneumothorax_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/pneumothorax_gradcam"},
    "Consolidation": {
        "diffusion":   f"{DIFFUSION_ROOT}/consolidation-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/consolidation_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/consolidation_gradcam"},
    "Nodule/Mass": {
        "diffusion":   f"{DIFFUSION_ROOT}/nodule-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/nodule_mass_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/nodule_mass_gradcam"},
    "Cardiomegaly": {
        "diffusion":   f"{DIFFUSION_ROOT}/cardiomegaly-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/cardiomegaly_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/cardiomegaly_gradcam"},
    "Atelectasis": {
        "diffusion":   f"{DIFFUSION_ROOT}/atelectasis-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/atelectasis_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/atelectasis_gradcam"},
}
'''
TEST_MAP_DIRS = {
    "Consolidation": {"diffusion": str('/kaggle/input/datasets/kartikichandratre/ext-step7-lora-rank8/finetuned_maps'), "mcd_logit": None, "mcd_gradcam": None},
    "Cardiomegaly":  {"diffusion": str('/kaggle/input/datasets/kartikichandratre/ext-step7-lora-rank8/finetuned_maps'), "mcd_logit": None, "mcd_gradcam": None},
}
OUTPUT_DIR = "/kaggle/working/stage9_ft"
'''


# Plot and table ordering. Grad-CAM last because it is the skyline, not a peer.
METHOD_ORDER = ["diffusion", "mcd_logit", "mcd_gradcam"]
METHOD_COLORS = {"diffusion": "#3b6ea5", "mcd_logit": "#a5453b",
                 "mcd_gradcam": "#6b8f47"}

# Methods that are label-supervised. Reported, but never described as
# baselines that diffusion "should" beat.
SKYLINE_METHODS = {"mcd_gradcam"}

# Grad-CAM is signed in some implementations (ReLU'd in most). If your maps
# contain negatives and you want magnitude only, set True.
GRADCAM_ABS = False

# VinDr-CXR box annotations. Needs columns for image_id, class name, and
# x_min/y_min/x_max/y_max. Column names are remapped below if yours differ.
ANNOTATION_CSV = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_test.csv"
ANNOT_COLS = {"image_id": "image_id", "class_name": "class_name",
              "x_min": "x_min", "y_min": "y_min",
              "x_max": "x_max", "y_max": "y_max"}

METADATA_CSV_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_metadata.csv"

# Directories whose ids went into Stage 6 TRAINING. Used to assert these
# test maps were never trained on.
STAGE6_TRAIN_MAP_DIRS = [
    "/kaggle/input/datasets/pgc17ms072/pneumothorax-uncertainity-train-control",
    "/kaggle/input/datasets/kartikichandratre/atelectasis-uncertainity-train-eta1-n20-g1",
    "/kaggle/input/datasets/pgc17ms072/nodule-uncertainity-train-eta1-n20-g1",
    "/kaggle/input/datasets/pgc17ms072/consolidation-gen-train",
    "/kaggle/input/datasets/pgc17ms072/cardiomegaly-uncertainity-train-eta1-n20-g1-sess0",
    "/kaggle/input/datasets/pgc17ms072/cardiomegaly-unceratainty-train-eta1-n20-g1-sess1",
    "/kaggle/input/datasets/kartikichandratre/cardiomelgaly-uncertainity-sess2",
]


# Boxes were drawn on the ORIGINAL DICOM resolution. Maps are at 512.
# If your annotation CSV is already rescaled to 512, set this False.
BOXES_IN_ORIGINAL_RESOLUTION = True
# width/height in ANNOTATION_CSV are named 'columns' and 'rows'
# respectively (image array dimensions), not 'original_width'/'height'.
ORIGINAL_SIZE_COLS = {"width": "columns", "height": "rows"}

MAP_SIZE = 512
DICE_PERCENTILES = [90.0, 95.0, 99.0]
N_BOOTSTRAP = 2000
ALPHA = 0.05
RANDOM_SEED = 42

OUTPUT_DIR = "/kaggle/working/stage9_testset_maps"

# Candidate array keys inside a SINGLE-MAP .npz. First match wins.
VARIANCE_KEYS = ["variance", "var", "pixel_variance", "sigma2",
                 "variance_map", "uncertainty", "std", "sigma",
                 # Grad-CAM / saliency naming
                 "cam", "gradcam", "grad_cam", "heatmap", "saliency",
                 "attribution", "map"]

# CHUNKED .npz support. Stage 4 writes many images per file, keyed
# "{image_id}__variance" and "{image_id}__mean". The separator and the
# suffixes that identify the array we want are configurable because the
# diffusion and Grad-CAM writers may not use the same convention.
CHUNK_SEPARATOR = "__"
CHUNK_VALUE_SUFFIXES = ["variance", "var", "uncertainty", "sigma2",
                        "cam", "gradcam", "heatmap", "saliency"]
# Suffixes present in the file but NOT to be scored (e.g. the ensemble mean).
CHUNK_IGNORE_SUFFIXES = ["mean", "std_err", "count", "n", "z_t"]


# =============================================================================
# STEP 0 — inspect. RUN THIS ALONE FIRST.
# =============================================================================
def inspect_maps(n_show=3):
    """Print what is actually inside the map files. Do this before anything."""
    for class_name, paths in TEST_MAP_DIRS.items():
        for kind in METHOD_ORDER:
            d = paths.get(kind)
            if d is None:
                print(f"{class_name:15s} {kind:12s} -- not provided")
                continue
            p = Path(d)
            if not p.exists():
                print(f"{class_name:15s} {kind:12s} -- MISSING DIR {d}")
                continue
            files = sorted(p.glob("*.npz")) + sorted(p.glob("*.npy"))
            print(f"\n{class_name:15s} {kind:12s} {len(files)} file(s) in {d}")
            for f in files[:n_show]:
                if f.suffix == ".npy":
                    a = np.load(f)
                    print(f"    {f.name}: npy {a.shape} {a.dtype}")
                    continue
                with np.load(f, allow_pickle=True) as z:
                    keys = list(z.files)
                    chunked = any(CHUNK_SEPARATOR in k for k in keys)
                    if chunked:
                        ids = sorted({k.rpartition(CHUNK_SEPARATOR)[0] for k in keys})
                        sufs = sorted({k.rpartition(CHUNK_SEPARATOR)[2].lower()
                                       for k in keys})
                        shape = z[keys[0]].shape
                        scored = [s for s in sufs if s in CHUNK_VALUE_SUFFIXES]
                        ignored = [s for s in sufs if s in CHUNK_IGNORE_SUFFIXES]
                        unknown = [s for s in sufs if s not in CHUNK_VALUE_SUFFIXES
                                   and s not in CHUNK_IGNORE_SUFFIXES]
                        print(f"    {f.name}: CHUNKED, {len(ids)} image(s), "
                              f"shape {shape}")
                        print(f"        suffixes  scored={scored}  "
                              f"ignored={ignored}  UNKNOWN={unknown}")
                        print(f"        example id: {ids[0]}")
                        if unknown:
                            print("        ^ decide whether any UNKNOWN suffix is "
                                  "the map, then update CHUNK_VALUE_SUFFIXES")
                    else:
                        print(f"    {f.name}: SINGLE-MAP, keys="
                              f"{ {k: z[k].shape for k in keys[:6]} }")
    print("\nWhen every row looks right, call main().")


# =============================================================================
# Loading
# =============================================================================
def _postprocess(m, method):
    """Reduce an ensemble stack if needed, apply Grad-CAM abs, resize."""
    if m.ndim == 3:
        warnings.warn(f"array is {m.shape}; taking per-pixel variance over axis 0.")
        m = m.var(axis=0)
    m = np.squeeze(m).astype(np.float64)
    if method == "mcd_gradcam" and GRADCAM_ABS:
        m = np.abs(m)
    if m.shape != (MAP_SIZE, MAP_SIZE):
        # Latent (64x64) and Grad-CAM (~16x16) maps upsample bilinearly.
        m = zoom(m, (MAP_SIZE / m.shape[0], MAP_SIZE / m.shape[1]), order=1)
    return m


def _read_chunked(z, path):
    """
    Chunked layout: many images per file, keys '{image_id}__{suffix}'.
    Returns {image_id: raw_array} for the value suffixes only, so the
    companion '__mean' arrays are never scored as if they were variance.
    """
    out, skipped = {}, set()
    for key in z.files:
        if CHUNK_SEPARATOR not in key:
            continue
        image_id, _, suffix = key.rpartition(CHUNK_SEPARATOR)
        s = suffix.lower()
        if s in CHUNK_IGNORE_SUFFIXES:
            continue
        if s not in CHUNK_VALUE_SUFFIXES:
            skipped.add(s)
            continue
        if image_id in out:
            raise KeyError(
                f"{path.name}: image_id '{image_id}' has more than one scorable "
                f"suffix. Narrow CHUNK_VALUE_SUFFIXES so exactly one matches."
            )
        out[image_id] = z[key]
    if skipped:
        warnings.warn(f"{path.name}: ignored unrecognised suffixes {sorted(skipped)}. "
                      f"Add to CHUNK_VALUE_SUFFIXES if one of them is the map.")
    return out


def _read_single(z, path):
    for k in VARIANCE_KEYS:
        if k in z.files:
            return {path.stem: z[k]}
    raise KeyError(
        f"{path.name} has keys {list(z.files)[:8]}, none in VARIANCE_KEYS and "
        f"none matching the chunked '{{id}}{CHUNK_SEPARATOR}{{suffix}}' pattern. "
        f"Run inspect_maps() and update the config."
    )


def load_map_dir(d, method=None):
    """image_id -> map at MAP_SIZE. Handles chunked and per-file layouts."""
    out, native = {}, []
    files = sorted(Path(d).glob("*.npz")) + sorted(Path(d).glob("*.npy"))
    for f in files:
        if f.suffix == ".npy":
            raw = {f.stem: np.load(f)}
        else:
            with np.load(f, allow_pickle=True) as z:
                chunked = any(CHUNK_SEPARATOR in k for k in z.files)
                raw = _read_chunked(z, f) if chunked else _read_single(z, f)
                raw = {k: np.array(v) for k, v in raw.items()}
        for image_id, arr in raw.items():
            if image_id in out:
                warnings.warn(f"duplicate image_id '{image_id}' in {d}; keeping first.")
                continue
            native.append(np.squeeze(arr).shape[0])
            out[image_id] = _postprocess(arr, method)
    return out, (float(np.median(native)) if native else np.nan)


def explode_boxes(df):
    """
    ANNOTATION_CSV is one row per IMAGE, with all boxes for that image
    packed into a 'boxes' column as a Python-literal string:
        "[{'class_name': 'Cardiomegaly', 'x_min': 863.0, ...}, {...}]"
    build_box_masks() expects one row per BOX (the ANNOT_COLS schema).
    This explodes wide -> long so the rest of the pipeline is unchanged.
    Images with no boxes, or an unparsable field, contribute nothing --
    correct, since there is no annotation to score against.
    """
    records = []
    for _, row in df.iterrows():
        raw = row.get("boxes")
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            continue
        try:
            boxes = literal_eval(raw) if isinstance(raw, str) else raw
        except (ValueError, SyntaxError):
            continue
        for b in boxes:
            records.append({
                "image_id": row["image_id"],
                "class_name": b.get("class_name"),
                "x_min": b.get("x_min"), "y_min": b.get("y_min"),
                "x_max": b.get("x_max"), "y_max": b.get("y_max"),
                "rad_id": b.get("rad_id"),
            })
    out = pd.DataFrame(records)
    n_classes = out["class_name"].nunique() if len(out) else 0
    print(f"  exploded {len(df)} image row(s) -> {len(out)} box row(s) "
          f"across {n_classes} classes")
    return out


def build_box_masks(annotations, metadata, class_name, image_ids):
    """image_id -> boolean mask of union of radiologist boxes for this class."""
    c = ANNOT_COLS
    sub = annotations[annotations[c["class_name"]] == class_name]
    sizes = {}
    if BOXES_IN_ORIGINAL_RESOLUTION:
        w, h = ORIGINAL_SIZE_COLS["width"], ORIGINAL_SIZE_COLS["height"]
        src = annotations if w in annotations.columns else metadata
        if w not in src.columns:
            raise KeyError(
                f"BOXES_IN_ORIGINAL_RESOLUTION=True but no '{w}' column in the "
                f"annotation or metadata CSV. Either add it or set the flag False."
            )
        sizes = {str(r[c["image_id"]] if c["image_id"] in src.columns else r["image_id"]):
                 (float(r[w]), float(r[h])) for _, r in src.iterrows()}

    masks = {}
    for image_id in image_ids:
        rows = sub[sub[c["image_id"]].astype(str) == str(image_id)]
        if rows.empty:
            continue
        mask = np.zeros((MAP_SIZE, MAP_SIZE), dtype=bool)
        sx = sy = 1.0
        if BOXES_IN_ORIGINAL_RESOLUTION:
            if str(image_id) not in sizes:
                continue
            ow, oh = sizes[str(image_id)]
            sx, sy = MAP_SIZE / ow, MAP_SIZE / oh
        for _, r in rows.iterrows():
            vals = [r[c["x_min"]], r[c["y_min"]], r[c["x_max"]], r[c["y_max"]]]
            if any(pd.isna(v) for v in vals):
                continue
            x0 = int(np.clip(round(r[c["x_min"]] * sx), 0, MAP_SIZE - 1))
            x1 = int(np.clip(round(r[c["x_max"]] * sx), 0, MAP_SIZE))
            y0 = int(np.clip(round(r[c["y_min"]] * sy), 0, MAP_SIZE - 1))
            y1 = int(np.clip(round(r[c["y_max"]] * sy), 0, MAP_SIZE))
            if x1 > x0 and y1 > y0:
                mask[y0:y1, x0:x1] = True
        if mask.any() and not mask.all():
            masks[str(image_id)] = mask
    return masks


# =============================================================================
# Per-image metrics
# =============================================================================
def dice(binary_map, mask):
    inter = np.logical_and(binary_map, mask).sum()
    denom = binary_map.sum() + mask.sum()
    return float(2 * inter / denom) if denom else np.nan


def metrics_for_image(var_map, mask):
    y = mask.ravel().astype(np.uint8)
    s = var_map.ravel().astype(np.float64)
    if not np.isfinite(s).all():
        s = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
    if y.sum() == 0 or y.sum() == len(y):
        return None

    row = {
        "pixel_auroc": float(roc_auc_score(y, s)),
        "pixel_auprc": float(average_precision_score(y, s)),
        "box_fraction": float(y.mean()),
    }
    row["auprc_lift"] = row["pixel_auprc"] / row["box_fraction"]
    for p in DICE_PERCENTILES:
        row[f"dice_p{p:g}"] = dice(var_map >= np.percentile(var_map, p), mask)
    flat = int(np.argmax(s))
    row["pointing_hit"] = int(mask.ravel()[flat])
    return row


# =============================================================================
# Aggregation
# =============================================================================
def bootstrap_mean_ci(values, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    v = np.asarray([x for x in values if np.isfinite(x)], dtype=float)
    if len(v) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = np.array([rng.choice(v, size=len(v), replace=True).mean()
                      for _ in range(n_boot)])
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def paired_compare(per_image, class_name, metric, method_a, method_b):
    """Paired Wilcoxon on the images both methods cover."""
    sub = per_image[(per_image["class_name"] == class_name)]
    a_df = sub[sub["method"] == method_a][["image_id", metric]]
    b_df = sub[sub["method"] == method_b][["image_id", metric]]
    merged = a_df.merge(b_df, on="image_id", suffixes=("_a", "_b")).dropna()
    if len(merged) < 6:
        return None
    a, b = merged[f"{metric}_a"].to_numpy(), merged[f"{metric}_b"].to_numpy()
    if np.allclose(a, b):
        return None
    stat, p = stats.wilcoxon(a, b)
    mean_d, lo, hi = bootstrap_mean_ci(a - b)
    # Rank-biserial correlation: an effect size that survives non-normality.
    d = a - b
    nz = d[d != 0]
    rbc = float(np.sign(nz).mean()) if len(nz) else np.nan
    return {"class_name": class_name, "metric": metric,
            "method_a": method_a, "method_b": method_b,
            "n_paired": len(merged),
            "mean_a": float(a.mean()), "mean_b": float(b.mean()),
            "mean_delta": mean_d, "delta_ci_low": lo, "delta_ci_high": hi,
            "rank_biserial": rbc,
            "a_wins_frac": float((a > b).mean()),
            "wilcoxon_stat": float(stat), "p_wilcoxon": float(p),
            "a_is_skyline": method_a in SKYLINE_METHODS,
            "b_is_skyline": method_b in SKYLINE_METHODS}


def friedman_omnibus(per_image, class_name, metric, methods):
    """
    Friedman test on images covered by ALL methods. This is the omnibus that
    licenses the post-hoc pairwise tests; without it, three pairwise
    Wilcoxons per class inflate the family-wise error rate.
    """
    sub = per_image[per_image["class_name"] == class_name]
    wide = sub.pivot_table(index="image_id", columns="method", values=metric)
    present = [m for m in methods if m in wide.columns]
    if len(present) < 3:
        return None
    wide = wide[present].dropna()
    if len(wide) < 6:
        return None
    stat, p = stats.friedmanchisquare(*[wide[m].to_numpy() for m in present])
    row = {"class_name": class_name, "metric": metric,
           "n_complete_cases": len(wide), "methods": ",".join(present),
           "friedman_stat": float(stat), "p_friedman": float(p)}
    # Mean rank per method (1 = best), the natural Friedman effect summary.
    ranks = wide.rank(axis=1, ascending=False)
    for m in present:
        row[f"mean_rank_{m}"] = float(ranks[m].mean())
    return row


# =============================================================================
# Contamination checks — both directions
# =============================================================================
def assert_split_integrity(metadata, all_map_ids):
    test_ids = set(metadata[metadata["split"] == "test"]["image_id"].astype(str))
    train_ids = set(metadata[metadata["split"] == "train"]["image_id"].astype(str))

    not_test = all_map_ids - test_ids
    if not_test:
        raise AssertionError(
            f"{len(not_test)} map(s) are NOT test-split images, e.g. "
            f"{sorted(not_test)[:5]}. These maps must come from the held-out "
            f"split or Approach A proves nothing."
        )
    print(f"  FORWARD: all {len(all_map_ids)} maps are test-split images.")

    trained = set()
    for d in STAGE6_TRAIN_MAP_DIRS:
        p = Path(d)
        if not p.exists():
            print(f"  WARNING: Stage 6 train dir missing, cannot verify: {d}")
            continue
        trained |= {f.stem for f in p.glob("*.npz")} | {f.stem for f in p.glob("*.npy")}
    overlap = all_map_ids & trained
    if overlap:
        raise AssertionError(
            f"{len(overlap)} test map id(s) also appear in Stage 6 training "
            f"inputs, e.g. {sorted(overlap)[:5]}. Split is broken."
        )
    print(f"  REVERSE: no overlap with {len(trained)} Stage 6 training ids.")
    print(f"  (test split holds {len(test_ids)} images; train {len(train_ids)})")


# =============================================================================
# Figures
# =============================================================================
def plot_auroc_distributions(per_image, out_dir):
    classes = sorted(per_image["class_name"].unique())
    fig, ax = plt.subplots(figsize=(1.6 * len(classes) + 3, 4.4))
    data, labels, colors = [], [], []
    present = [m for m in METHOD_ORDER if m in set(per_image["method"])]
    for c in classes:
        for method in present:
            v = per_image[(per_image["class_name"] == c) &
                          (per_image["method"] == method)]["pixel_auroc"].dropna()
            if len(v):
                data.append(v.to_numpy())
                star = "*" if method in SKYLINE_METHODS else ""
                labels.append(f"{c}\n{method}{star}\nn={len(v)}")
                colors.append(METHOD_COLORS.get(method, "#888888"))
    bp = ax.boxplot(data, labels=labels, showmeans=True, patch_artist=True)
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.55)
    ax.axhline(0.5, color="k", ls="--", lw=1.2, label="chance")
    ax.set_ylabel("Per-image pixel AUROC")
    ax.set_title("Held-out test split: does uncertainty localise the radiologist boxes?")
    ax.set_xlabel("* label-supervised skyline, not a peer baseline", fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis="y")
    plt.setp(ax.get_xticklabels(), fontsize=7)
    plt.tight_layout()
    plt.savefig(Path(out_dir) / "stage9_auroc_distributions.png", dpi=150, bbox_inches="tight")
    plt.close()


# =============================================================================
# Orchestration
# =============================================================================
def main():
    out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
    rows, summary_rows, paired_rows = [], [], []

    print("[1/5] Loading annotations and metadata...")
    annotations = pd.read_csv(ANNOTATION_CSV)
    annotations = explode_boxes(annotations)
    metadata = pd.read_csv(METADATA_CSV_PATH)
    if isinstance(metadata["labels"].iloc[0], str):
        metadata["labels"] = metadata["labels"].apply(literal_eval)

    print("[2/5] Loading maps...")
    loaded, native_res = {}, []
    all_ids = set()
    for class_name, paths in TEST_MAP_DIRS.items():
        loaded[class_name] = {}
        for method in METHOD_ORDER:
            d = paths.get(method)
            if d is None or not Path(d).exists():
                print(f"  {class_name:15s} {method:12s} -- MISSING {d}")
                continue
            maps, res = load_map_dir(d, method=method)
            if not maps:
                continue
            loaded[class_name][method] = maps
            all_ids |= set(maps.keys())
            native_res.append({"class_name": class_name, "method": method,
                               "n_maps": len(maps), "median_native_size": res})
            print(f"  {class_name:15s} {method:12s} {len(maps):4d} maps "
                  f"(native {res:g}px -> {MAP_SIZE}px)")
    if not all_ids:
        raise RuntimeError("No maps loaded. Run inspect_maps() and fix TEST_MAP_DIRS.")

    print("[3/5] Split integrity...")
    assert_split_integrity(metadata, all_ids)

    print("[4/5] Per-image metrics...")
    for class_name, methods in loaded.items():
        masks = build_box_masks(annotations, metadata, class_name,
                                set().union(*[set(m) for m in methods.values()]))
        print(f"  {class_name}: {len(masks)} image(s) with usable boxes")
        for method, maps in methods.items():
            for image_id, var_map in tqdm(maps.items(), desc=f"{class_name}/{method}",
                                          leave=False):
                if image_id not in masks:
                    continue
                m = metrics_for_image(var_map, masks[image_id])
                if m is None:
                    continue
                m.update({"class_name": class_name, "method": method,
                          "image_id": image_id})
                rows.append(m)

    per_image = pd.DataFrame(rows)
    if per_image.empty:
        raise RuntimeError(
            "No image matched a box mask. Most likely ANNOT_COLS, the class "
            "name spelling, or BOXES_IN_ORIGINAL_RESOLUTION is wrong."
        )
    per_image.to_csv(out / "stage9_per_image_metrics.csv", index=False)

    metric_cols = (["pixel_auroc", "pixel_auprc", "auprc_lift", "pointing_hit"]
                   + [f"dice_p{p:g}" for p in DICE_PERCENTILES])

    for (class_name, method), g in per_image.groupby(["class_name", "method"]):
        row = {"class_name": class_name, "method": method, "n_images": len(g),
               "mean_box_fraction": g["box_fraction"].mean()}
        for mc in metric_cols:
            mean, lo, hi = bootstrap_mean_ci(g[mc])
            row[f"{mc}_mean"], row[f"{mc}_ci_low"], row[f"{mc}_ci_high"] = mean, lo, hi
        v = g["pixel_auroc"].dropna().to_numpy()
        if len(v) >= 6:
            _, p = stats.wilcoxon(v - 0.5)
            row["p_vs_chance"] = float(p)
            row["frac_above_chance"] = float((v > 0.5).mean())
        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    ok = summary["p_vs_chance"].notna()
    summary.loc[ok, "p_vs_chance_bh"] = multipletests(
        summary.loc[ok, "p_vs_chance"], alpha=ALPHA, method="fdr_bh")[1]
    summary.to_csv(out / "stage9_summary.csv", index=False)

    print("\n" + "=" * 78)
    print("HELD-OUT TEST SPLIT — localisation")
    print("=" * 78)
    print(summary[["class_name", "method", "n_images", "pixel_auroc_mean",
                   "pixel_auroc_ci_low", "pixel_auroc_ci_high",
                   "pointing_hit_mean", "p_vs_chance_bh"]].round(4).to_string(index=False))

    # Resolution audit -- the confound that can explain a Grad-CAM advantage.
    res_df = pd.DataFrame(native_res)
    res_df.to_csv(out / "stage9_native_resolutions.csv", index=False)
    print("\nNative map resolution before upsampling (read alongside any gap):")
    print(res_df.pivot_table(index="class_name", columns="method",
                             values="median_native_size").to_string())

    print("\n[5/6] Friedman omnibus across the three methods...")
    omni_rows = []
    for class_name in loaded:
        for mc in ["pixel_auroc", "pointing_hit"]:
            r = friedman_omnibus(per_image, class_name, mc, METHOD_ORDER)
            if r:
                omni_rows.append(r)
    if omni_rows:
        omni = pd.DataFrame(omni_rows)
        omni["p_friedman_bh"] = multipletests(omni["p_friedman"], alpha=ALPHA,
                                              method="fdr_bh")[1]
        omni.to_csv(out / "stage9_friedman_omnibus.csv", index=False)
        cols = ["class_name", "metric", "n_complete_cases", "friedman_stat",
                "p_friedman_bh"] + [f"mean_rank_{m}" for m in METHOD_ORDER
                                    if f"mean_rank_{m}" in omni.columns]
        print(omni[cols].round(4).to_string(index=False))
        print("  mean_rank: 1 = best of the three on that image. ")
    else:
        print("  Skipped -- no class has all three methods on >=6 shared images.")

    print("\n[6/6] Post-hoc pairwise comparisons...")
    method_pairs = [("diffusion", "mcd_logit"),
                    ("diffusion", "mcd_gradcam"),
                    ("mcd_logit", "mcd_gradcam")]
    for class_name, methods in loaded.items():
        for ma, mb in method_pairs:
            if ma not in methods or mb not in methods:
                continue
            for mc in ["pixel_auroc", "pointing_hit"]:
                r = paired_compare(per_image, class_name, mc, ma, mb)
                if r:
                    paired_rows.append(r)

    if paired_rows:
        paired = pd.DataFrame(paired_rows)
        paired["p_bh"] = multipletests(paired["p_wilcoxon"], alpha=ALPHA,
                                       method="fdr_bh")[1]
        paired["comparison"] = paired["method_a"] + " vs " + paired["method_b"]
        paired.to_csv(out / "stage9_pairwise_comparisons.csv", index=False)
        for mc in ["pixel_auroc", "pointing_hit"]:
            g = paired[paired["metric"] == mc]
            if g.empty:
                continue
            print(f"\n  --- {mc} ---")
            print(g[["class_name", "comparison", "n_paired", "mean_a", "mean_b",
                     "mean_delta", "delta_ci_low", "delta_ci_high",
                     "rank_biserial", "p_bh"]].round(4).to_string(index=False))

        print("\nINTERPRETATION GUARD")
        print("  diffusion vs mcd_logit   -- the fair comparison. Both are")
        print("      uncertainty estimates; on this split neither model has")
        print("      seen these images. This is what Stage 4 could not do.")
        print("  diffusion vs mcd_gradcam -- NOT a fair comparison. Grad-CAM is")
        print("      class-evidence attribution from a label-supervised model,")
        print("      i.e. it is optimised for the very thing being scored.")
        print("      Report it as a skyline. A gap here is expected and is not")
        print("      evidence that diffusion uncertainty failed.")
        print("  Check the resolution table above before reading any gap as")
        print("      method quality: coarse upsampled maps flatter box overlap.")
    else:
        print("  No class has two or more methods -- skipped.")

    plot_auroc_distributions(per_image, out)
    print(f"\nSaved to {out}")
    print("\nNOTES FOR THE WRITE-UP")
    print("  1. Metrics are per-image, bootstrapped over images. If the Stage 1")
    print("     train numbers pooled pixels, recompute them this way before")
    print("     quoting train and test together.")
    print("  2. The test split removes the IMAGE-level confound from Stage 4.")
    print("     It does not remove LABEL supervision: both MCD variants come")
    print("     from a classifier trained on these classes. Diffusion saw")
    print("     neither the images nor the labels.")
    print("  3. These CIs cover image sampling only. The maps are themselves")
    print("     stochastic (N=20 reconstructions); a second generation seed")
    print("     would move them by an unmeasured amount.")
    return per_image, summary


if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    inspect_maps()
    print("\nInspection done. Call main() once VARIANCE_KEYS and paths are set.")


Consolidation   diffusion    3 file(s) in /kaggle/input/datasets/kartikichandratre/ext-step7-lora-rank8/finetuned_maps
    uncertainty_chunk_000.npz: CHUNKED, 50 image(s), shape (512, 512)
        suffixes  scored=['variance']  ignored=['mean', 'z_t']  UNKNOWN=[]
        example id: 008b3176a7248a0a189b5731ac8d2e95
    uncertainty_chunk_001.npz: CHUNKED, 50 image(s), shape (512, 512)
        suffixes  scored=['variance']  ignored=['mean', 'z_t']  UNKNOWN=[]
        example id: 5714aea8b5a2d9b030196646842a6d47
    uncertainty_chunk_002.npz: CHUNKED, 50 image(s), shape (512, 512)
        suffixes  scored=['variance']  ignored=['mean', 'z_t']  UNKNOWN=[]
        example id: a5a2a3b02ccb9c3145d553d269e4b0b8
Consolidation   mcd_logit    -- not provided
Consolidation   mcd_gradcam  -- not provided

Cardiomegaly    diffusion    3 file(s) in /kaggle/input/datasets/kartikichandratre/ext-step7-lora-rank8/finetuned_maps
    uncertainty_chunk_000.npz: CHUNKED, 50 image(s), shape (512, 512)
      

In [3]:
DRIVER_MODE = True
"""
step1_generation_adapted.py
============================================================================
CELL 3. Generation core, adapted DIRECTLY from your working notebook
(atelectasis-gen-eta1-n20-g1.ipynb). This supersedes the earlier
reconstruction, which had three errors this file corrects:

  1. H5 stores float32 ALREADY in [-1,1]. The reconstruction divided by
     255 and rescaled, which would have fed the VAE garbage.
  2. Real config is 30 inversion / 30 inference steps, not 50/15.
  3. Inversion uses diffusers DDIMInverseScheduler, not a hand-rolled
     update. Your version is correct; mine was an approximation.

WHAT CHANGED FROM YOUR NOTEBOOK
Exactly one thing: load_pipeline() takes an optional lora_path. Every
other function is byte-equivalent in behaviour to yours -- same Welford
accumulator, same sequential N=20, same per-finding prompt, same
attention slicing, same fp16, same seed_base=0 so generators are
manual_seed(0..19).

WHY THE PROMPT STAYS LABEL-CONDITIONED
Your notebook uses prompt = f"a chest x-ray showing {finding.lower()}".
That is kept EXACTLY as-is. The fine-tuning experiment changes UNet
weights only. If the prompt convention also changed, the comparison would
confound weight adaptation with conditioning change. The prompt asymmetry
exists identically in baseline and fine-tuned arms, so it cancels.

(Note for the viva, not for this script: this means the generator does
receive class-name conditioning at inference, which is a weaker form of
label access than supervised training but not zero. Thesis 4.2 describes
the weights as label-free, which is accurate; it describes the estimator
as label-free, which is stronger than what the prompt convention
supports.)

MEASURED TIMING (from your own diagnostic cells, not estimated)
  72.4 s/image single-GPU
  154.5 s for 4 images across 2x T4  ->  ~38.6 s/image effective
  => 54 images  ~35 min
     114 images ~73 min
"""

import ast
import gc
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler, DDIMInverseScheduler
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# =============================================================================
# CONFIG -- unchanged from your notebook
# =============================================================================
H5_PATH = Path("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5")
METADATA_CSV = Path("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_test.csv")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_INVERSION_STEPS = 30
NUM_INFERENCE_STEPS = 30
NUM_ENSEMBLE = 20
ETA = 1
GUIDANCE_SCALE = 1.0
IMG_SIZE = 512
CHUNK_SIZE = 50

# LoRA weights directory, or None for baseline. Set before calling
# load_all_pipelines(). This is the ONLY addition to your pipeline.
LORA_PATH = None

NUM_GPUS = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(NUM_GPUS)] if NUM_GPUS > 0 else ["cpu"]


def _discover_models():
    local = []
    inp = Path("/kaggle/input")
    if inp.exists():
        for p in inp.rglob("model_index.json"):
            local.append(str(p.parent))
    return local + ["stanfordmimi/RoentGen-v2", "runwayml/stable-diffusion-v1-5"]


MODEL_CANDIDATES = _discover_models()


# =============================================================================
# Pipeline -- your loader, plus lora_path
# =============================================================================
def load_pipeline(device, lora_path=None):
    pipe = None
    for model_id in MODEL_CANDIDATES:
        try:
            is_local = Path(model_id).exists()
            kwargs = {"torch_dtype": torch.float16, "safety_checker": None,
                      "requires_safety_checker": False}
            if is_local:
                kwargs["local_files_only"] = True
            else:
                tok = globals().get("HF_TOKEN", None)
                if tok:
                    kwargs["token"] = tok
            pipe = StableDiffusionPipeline.from_pretrained(model_id, **kwargs)
            print(f"[{device}] loaded {model_id}")
            break
        except Exception as e:
            print(f"[{device}] could not load {model_id}: {e}")
    if pipe is None:
        raise RuntimeError("No candidate model could be loaded.")

    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    # --- THE ONLY ADDITION TO YOUR PIPELINE ---
    if lora_path is not None:
        pipe.load_lora_weights(str(lora_path))
        pipe.fuse_lora()          # fuse for inference speed; no-op on baseline
        print(f"[{device}] LoRA fused from {lora_path}")
    # ------------------------------------------

    pipe.enable_attention_slicing()
    pipe.vae.enable_slicing()
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    for m in (pipe.unet, pipe.vae, pipe.text_encoder):
        m.eval()
        for p in m.parameters():
            p.requires_grad_(False)
    return pipe


def load_all_pipelines(lora_path=None):
    pipes = {d: load_pipeline(d, lora_path) for d in DEVICES}
    scheds = {d: DDIMScheduler.from_config(pipes[d].scheduler.config) for d in DEVICES}
    inv_scheds = {d: DDIMInverseScheduler.from_config(pipes[d].scheduler.config)
                  for d in DEVICES}
    return pipes, scheds, inv_scheds


# =============================================================================
# Core -- verbatim behaviour from your notebook
# =============================================================================
@torch.no_grad()
def load_image_tensor_from_h5(h5_path, image_id):
    """H5 holds float32 (512,512) ALREADY in [-1,1]. No rescaling."""
    with h5py.File(h5_path, "r") as h5f:
        if image_id not in h5f:
            raise KeyError(f"{image_id} not found in {h5_path}")
        arr = h5f[image_id][:]
    t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    return t.repeat(1, 3, 1, 1)


@torch.no_grad()
def encode_to_latent(pipe, image_tensor):
    image_tensor = image_tensor.to(device=pipe.device, dtype=pipe.unet.dtype)
    dist = pipe.vae.encode(image_tensor).latent_dist
    return dist.mean * pipe.vae.config.scaling_factor


@torch.no_grad()
def get_text_embeddings(pipe, prompt):
    tokens = pipe.tokenizer(
        prompt, padding="max_length", max_length=pipe.tokenizer.model_max_length,
        truncation=True, return_tensors="pt").input_ids.to(pipe.device)
    return pipe.text_encoder(tokens)[0].to(dtype=pipe.unet.dtype)


@torch.no_grad()
def ddim_invert(latents, pipe, inverse_scheduler, text_embeddings,
                num_inversion_steps=NUM_INVERSION_STEPS):
    inverse_scheduler.set_timesteps(num_inversion_steps, device=latents.device)
    latents = latents.clone()
    for t in inverse_scheduler.timesteps:
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=text_embeddings).sample
        latents = inverse_scheduler.step(noise_pred, t, latents).prev_sample
    return latents


@torch.no_grad()
def ddim_sample_from_latent(z_T, pipe, scheduler, text_embeddings,
                            num_inference_steps=NUM_INFERENCE_STEPS,
                            eta=ETA, generator=None):
    scheduler.set_timesteps(num_inference_steps, device=z_T.device)
    latents = z_T.clone()
    for t in scheduler.timesteps:
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=text_embeddings).sample
        latents = scheduler.step(noise_pred, t, latents, eta=eta,
                                 generator=generator).prev_sample
    return latents


@torch.no_grad()
def decode_latents_to_grayscale(pipe, latents):
    latents = latents / pipe.vae.config.scaling_factor
    image = pipe.vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    return image.float().cpu().mean(dim=1).squeeze(0)


class WelfordAccumulator:
    def __init__(self, shape, device="cpu"):
        self.n = 0
        self.mean = torch.zeros(shape, dtype=torch.float32, device=device)
        self.M2 = torch.zeros(shape, dtype=torch.float32, device=device)

    def update(self, x):
        x = x.to(dtype=torch.float32, device=self.mean.device)
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        self.M2 += delta * (x - self.mean)

    @property
    def variance(self):
        if self.n < 2:
            return torch.zeros_like(self.mean)
        return self.M2 / (self.n - 1)


@torch.no_grad()
def run_uncertainty_ensemble(pipe, scheduler, z_T, text_embeddings,
                             num_ensemble=NUM_ENSEMBLE,
                             num_inference_steps=NUM_INFERENCE_STEPS,
                             eta=ETA, seed_base=0):
    acc = WelfordAccumulator(shape=(IMG_SIZE, IMG_SIZE), device="cpu")
    for i in range(num_ensemble):
        gen = torch.Generator(device=pipe.device).manual_seed(seed_base + i)
        lat = ddim_sample_from_latent(z_T, pipe, scheduler, text_embeddings,
                                      num_inference_steps=num_inference_steps,
                                      eta=eta, generator=gen)
        img = decode_latents_to_grayscale(pipe, lat)
        acc.update(img)
        del lat, img, gen
        torch.cuda.empty_cache()
    return acc.mean, acc.variance


@torch.no_grad()
def process_one_image(pipe, scheduler, inverse_scheduler, image_id, finding, device):
    # UNCHANGED from your notebook -- prompt keeps the finding name.
    prompt = f"a chest x-ray showing {finding.lower()}"

    image_tensor = load_image_tensor_from_h5(H5_PATH, image_id)
    emb = get_text_embeddings(pipe, prompt)
    lat0 = encode_to_latent(pipe, image_tensor)
    z_T = ddim_invert(lat0, pipe, inverse_scheduler, emb,
                      num_inversion_steps=NUM_INVERSION_STEPS)
    mean_img, var_img = run_uncertainty_ensemble(
        pipe, scheduler, z_T, emb, num_ensemble=NUM_ENSEMBLE,
        num_inference_steps=NUM_INFERENCE_STEPS, eta=ETA)

    out = {"mean": mean_img.numpy().astype(np.float32),
           "variance": var_img.numpy().astype(np.float32),
           "z_T": z_T.detach().to(dtype=torch.float16).cpu().numpy(),
           "finding": finding}
    del image_tensor, emb, lat0, z_T, mean_img, var_img
    torch.cuda.empty_cache()
    gc.collect()
    return out


def process_partition(pipe, scheduler, inverse_scheduler, partition_df, device):
    out = {}
    for _, row in tqdm(partition_df.iterrows(), total=len(partition_df),
                       desc=f"[{device}]", leave=False):
        try:
            out[row["image_id"]] = process_one_image(
                pipe, scheduler, inverse_scheduler,
                row["image_id"], row["finding"], device)
        except Exception as e:
            print(f"[{device}] FAILED on {row['image_id']}: {e}")
    return out


def process_chunk(chunk_df, pipes, scheds, inv_scheds):
    parts = np.array_split(chunk_df, len(DEVICES))
    results = {}
    with ThreadPoolExecutor(max_workers=len(DEVICES)) as ex:
        futures = {ex.submit(process_partition, pipes[DEVICES[i]],
                             scheds[DEVICES[i]], inv_scheds[DEVICES[i]],
                             parts[i], DEVICES[i]): DEVICES[i]
                   for i in range(len(DEVICES)) if len(parts[i]) > 0}
        for f in as_completed(futures):
            try:
                results.update(f.result())
            except Exception as e:
                print(f"[{futures[f]}] partition failed: {e}")
    return results


def save_chunk_npz(chunk_results, chunk_idx, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    save_dict, meta = {}, []
    for image_id, d in chunk_results.items():
        save_dict[f"{image_id}__mean"] = d["mean"]
        save_dict[f"{image_id}__variance"] = d["variance"]
        save_dict[f"{image_id}__z_T"] = d["z_T"]
        meta.append({"image_id": image_id, "finding": d["finding"],
                     "variance_mean": float(d["variance"].mean()),
                     "variance_max": float(d["variance"].max())})
    p = out_dir / f"uncertainty_chunk_{chunk_idx:03d}.npz"
    np.savez_compressed(p, **save_dict)
    pd.DataFrame(meta).to_csv(
        out_dir / f"uncertainty_chunk_{chunk_idx:03d}_meta.csv", index=False)
    print(f"  saved chunk {chunk_idx}: {len(chunk_results)} imgs -> "
          f"{p.name} ({p.stat().st_size / 1e6:.1f} MB)")


def build_target_df(findings):
    md = pd.read_csv(METADATA_CSV)
    md["labels_parsed"] = md["labels"].apply(ast.literal_eval)

    def first_match(labels):
        for lab in labels:
            if lab in findings:
                return lab
        return None

    md["finding"] = md["labels_parsed"].apply(first_match)
    tdf = md[md["finding"].notna()].reset_index(drop=True)[["image_id", "finding"]]
    print(f"targeting {len(tdf):,} images across {findings}")
    print(tdf["finding"].value_counts())
    return tdf


# =============================================================================
# STEP 5a -- REPRODUCIBILITY CHECK. Run this before the full generation.
# =============================================================================
def verify_baseline_reproduces(existing_map_dir, n_check=3):
    """
    Regenerates a few images with LORA_PATH=None and compares against your
    EXISTING baseline maps.

    Why this matters: if this environment reproduces your original run,
    fine-tuned maps can be compared directly against the existing baseline
    maps and you skip regenerating 54 baseline images (~35 min saved). If
    it does NOT reproduce -- different diffusers version, different GPU
    kernel -- you must regenerate both arms with this file, or the
    comparison confounds the LoRA effect with an environment difference.
    """
    from numpy.testing import assert_allclose

    maps, _ = load_map_dir(str(existing_map_dir), method="diffusion")
    ids = list(maps.keys())[:n_check]
    print(f"Re-generating {len(ids)} image(s) at baseline to compare...\n")

    pipes, scheds, inv_scheds = load_all_pipelines(lora_path=None)
    d = DEVICES[0]
    md = pd.read_csv(METADATA_CSV)
    md["labels_parsed"] = md["labels"].apply(ast.literal_eval)

    worst = 0.0
    for image_id in ids:
        row = md[md["image_id"] == image_id]
        if row.empty:
            print(f"  {image_id}: not in metadata, skipping")
            continue
        finding = [l for l in row.iloc[0]["labels_parsed"]][0]
        res = process_one_image(pipes[d], scheds[d], inv_scheds[d],
                                image_id, finding, d)
        old, new = maps[image_id], res["variance"]
        rel = np.abs(new - old).max() / (old.max() + 1e-12)
        worst = max(worst, rel)
        corr = np.corrcoef(old.ravel(), new.ravel())[0, 1]
        print(f"  {image_id}: max rel diff {rel:.4f}, corr {corr:.6f}")

    print(f"\n{'=' * 62}")
    if worst < 0.01:
        print("REPRODUCES. Skip baseline regeneration -- compare fine-tuned")
        print("maps directly against your existing Stage 9 baseline maps.")
    else:
        print(f"DOES NOT REPRODUCE (worst rel diff {worst:.4f}).")
        print("You MUST regenerate both arms with this file. Budget 2x the")
        print("generation time. Do not compare against the original maps.")
    print("=" * 62)
    return worst


if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    print(f"devices: {DEVICES}")
    print(f"models:  {MODEL_CANDIDATES[:2]}")
    print("\nMeasured timing from your diagnostics: ~38.6 s/image on 2x T4")
    print("  54 images  -> ~35 min")
    print("  114 images -> ~73 min")
    print("\nNext: run verify_baseline_reproduces(<your existing map dir>)")
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [14]:
from pathlib import Path
p = Path("/kaggle/input/roentgen-lora-cxr-rank8")
for x in sorted(p.rglob("*"))[:15]:
    print(x)


In [16]:
LORA_PATH = None   # baseline
worst = verify_baseline_reproduces(
    "/kaggle/input/datasets/kartikichandratre/consolidation-uncertainty-test",
    n_check=3)

Re-generating 3 image(s) at baseline to compare...



Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:0] loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:1] loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete
  01570ee44031e4ebab6031501293bf66: max rel diff 0.4538, corr 0.989715
  05d676834dbed1639cb5eea70c1e307b: max rel diff 0.2044, corr 0.999858
  0c5ff01c7bfb4362fcd98f36e555b08c: max rel diff 0.1992, corr 0.991651

DOES NOT REPRODUCE (worst rel diff 0.4538).
You MUST regenerate both arms with this file. Budget 2x the
generation time. Do not compare against the original maps.


In [18]:
from scipy.stats import spearmanr
from scipy.ndimage import sobel
from sklearn.metrics import roc_auc_score
import ast

VERIFY_METADATA_CSV ='/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_test.csv'
md = pd.read_csv(VERIFY_METADATA_CSV)
md["labels_parsed"] = md["labels"].apply(ast.literal_eval)
annot = explode_boxes(pd.read_csv(ANNOTATION_CSV))
maps, _ = load_map_dir(BASELINE_DIR, method="diffusion")

pipes, scheds, inv_scheds = load_all_pipelines(lora_path=None)
d = DEVICES[0]

with h5py.File(H5_PATH, "r") as h5f:
    for image_id in list(maps.keys())[:3]:
        row = md[md.image_id == image_id]
        finding = row.iloc[0]["labels_parsed"][0]
        new = process_one_image(pipes[d], scheds[d], inv_scheds[d],
                                image_id, finding, d)["variance"]
        old = maps[image_id]

        img = h5f[image_id][:]
        edges = np.hypot(sobel(img.astype(float), 0), sobel(img.astype(float), 1))
        rho_o = spearmanr(edges.ravel(), old.ravel())[0]
        rho_n = spearmanr(edges.ravel(), new.ravel())[0]

        mask = build_box_masks(annot, md, finding, [image_id]).get(image_id)
        if mask is not None:
            y = mask.ravel().astype(int)
            a_o = roc_auc_score(y, old.ravel())
            a_n = roc_auc_score(y, new.ravel())
            print(f"{image_id[:12]}  AUROC {a_o:.4f}->{a_n:.4f} (d={a_n-a_o:+.4f})"
                  f"   rho {rho_o:.4f}->{rho_n:.4f} (d={rho_n-rho_o:+.4f})")

  exploded 660 image row(s) -> 5563 box row(s) across 14 classes


NameError: name 'BASELINE_DIR' is not defined

In [8]:

"""
metric_reproducibility_check.py
============================================================================
Replaces the max-rel-diff check, which was the wrong test.

WHY THE PREVIOUS CHECK WAS WRONG
verify_baseline_reproduces() thresholded on max relative pixel difference
and returned "DOES NOT REPRODUCE (worst 0.4538)". But the correlations in
that same output were 0.9897 / 0.9999 / 0.9917 -- the maps are structurally
near-identical. A single worst-pixel statistic on a map spanning
5e-06 to 0.103 is dominated by fp16 nondeterminism across 20 stochastic
sampling passes. It says nothing about whether the DERIVED metrics are
stable.

Step 6 compares pixel AUROC and edge rho. Those are the quantities that
must reproduce. This script measures them directly.

DECISION RULE (pre-specified, do not adjust after seeing the numbers)
    all |dAUROC| < 0.01   -> metrics stable. Compare fine-tuned maps
                             against EXISTING baseline maps. Generate one
                             arm only.
    any |dAUROC| > 0.02   -> genuine instability. Regenerate both arms.
    in between            -> borderline; regenerate both. The cost of a
                             contaminated calibration test exceeds 1h40.

0.02 is roughly half the smallest per-class effect the pre-registered
slopes predict for a plausible delta-rho, so anything above it would
contaminate the comparison.

PREREQUISITES -- all must already be in scope:
    stage9_final.py          -> load_map_dir, explode_boxes, build_box_masks
    step1_generation_adapted -> load_all_pipelines, process_one_image,
                                DEVICES, H5_PATH
"""

import ast

import h5py
import numpy as np
import pandas as pd
from scipy.ndimage import sobel
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
DRIVER_MODE = True
# =============================================================================
# CONFIG -- set these three to match the arm you are verifying
# =============================================================================
BASELINE_DIR = ("/kaggle/input/datasets/kartikichandratre/consolidation-uncertainty-test")

# The class these maps belong to. Used for BOX LOOKUP ONLY.
# Do NOT infer this from labels_parsed[0]: your annotation file has 14
# classes over 5,563 boxes, so multi-label images are common and the first
# label is often not the class whose maps these are. Getting it wrong
# scores AUROC against the wrong boxes and fakes a reproducibility failure.
CLASS_FOR_BOXES = "Consolidation"

# Must be the TEST-split CSV -- these are test images.
VERIFY_METADATA_CSV = ("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_test.csv")

N_CHECK = 3
STABLE_THRESHOLD = 0.01
UNSTABLE_THRESHOLD = 0.02


def edge_rho(img, var_map):
    img = img.astype(np.float64)
    edges = np.hypot(sobel(img, axis=0), sobel(img, axis=1))
    return float(spearmanr(edges.ravel(), var_map.ravel())[0])


def main():
    print("Loading annotations and metadata...")
    md = pd.read_csv(VERIFY_METADATA_CSV)
    md["labels_parsed"] = md["labels"].apply(ast.literal_eval)
    annot = explode_boxes(pd.read_csv(ANNOTATION_CSV))

    maps, _ = load_map_dir(BASELINE_DIR, method="diffusion")
    ids = list(maps.keys())[:N_CHECK]
    print(f"  {len(maps)} baseline maps, checking {len(ids)}\n")

    print("Loading baseline pipeline (no LoRA)...")
    pipes, scheds, inv_scheds = load_all_pipelines(lora_path=None)
    d = DEVICES[0]

    rows = []
    with h5py.File(H5_PATH, "r") as h5f:
        for image_id in ids:
            row = md[md.image_id == image_id]
            if row.empty:
                print(f"  {image_id}: NOT IN {VERIFY_METADATA_CSV}")
                continue

            # Prompt must match the ORIGINAL run's convention, which used
            # build_target_df's first_match over the targeted findings.
            labels = row.iloc[0]["labels_parsed"]
            finding = (CLASS_FOR_BOXES if CLASS_FOR_BOXES in labels
                       else labels[0])

            new = process_one_image(pipes[d], scheds[d], inv_scheds[d],
                                    image_id, finding, d)["variance"]
            old = maps[image_id]

            img = h5f[image_id][:]
            r_old, r_new = edge_rho(img, old), edge_rho(img, new)

            mask = build_box_masks(annot, md, CLASS_FOR_BOXES,
                                   [image_id]).get(image_id)
            if mask is None:
                print(f"  {image_id}: no {CLASS_FOR_BOXES} box, skipping")
                continue

            y = mask.ravel().astype(int)
            a_old = roc_auc_score(y, old.ravel())
            a_new = roc_auc_score(y, new.ravel())
            corr = float(np.corrcoef(old.ravel(), new.ravel())[0, 1])

            rows.append({"image_id": image_id, "prompt_finding": finding,
                         "auroc_old": a_old, "auroc_new": a_new,
                         "d_auroc": a_new - a_old,
                         "rho_old": r_old, "rho_new": r_new,
                         "d_rho": r_new - r_old, "map_corr": corr})
            print(f"  {image_id[:12]}  AUROC {a_old:.4f}->{a_new:.4f} "
                  f"(d={a_new - a_old:+.4f})   rho {r_old:.4f}->{r_new:.4f} "
                  f"(d={r_new - r_old:+.4f})   corr={corr:.6f}")

    if not rows:
        print("\nINCONCLUSIVE -- zero images compared. Check "
              "VERIFY_METADATA_CSV and CLASS_FOR_BOXES. This is NOT a pass.")
        return None

    df = pd.DataFrame(rows)
    worst_auroc = df["d_auroc"].abs().max()
    worst_rho = df["d_rho"].abs().max()

    print("\n" + "=" * 70)
    print(f"n compared:        {len(df)}")
    print(f"worst |d AUROC|:   {worst_auroc:.4f}")
    print(f"worst |d rho|:     {worst_rho:.4f}")
    print(f"min map corr:      {df['map_corr'].min():.6f}")
    print("=" * 70)

    if worst_auroc < STABLE_THRESHOLD:
        print("METRICS STABLE.")
        print("  -> Generate the FINE-TUNED arm only (~1h40 for 154 images).")
        print("  -> Compare against your existing Stage 9 baseline maps.")
    elif worst_auroc > UNSTABLE_THRESHOLD:
        print("METRICS UNSTABLE.")
        print("  -> Regenerate BOTH arms with this code (~3h20).")
        print("  -> Do not compare against the original maps.")
        print("  -> Worth reporting: this quantifies generation-seed")
        print("     variability, which the thesis lists as unmeasured.")
    else:
        print("BORDERLINE. Regenerate both arms -- a contaminated")
        print("calibration test costs more than the extra 1h40.")
    print("=" * 70)
    return df


#if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    #result = main()
main()

Loading annotations and metadata...
  exploded 660 image row(s) -> 5563 box row(s) across 14 classes
  54 baseline maps, checking 3

Loading baseline pipeline (no LoRA)...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:0] loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:1] loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete
  01570ee44031  AUROC 0.7363->0.7364 (d=+0.0000)   rho 0.6207->0.6207 (d=-0.0000)   corr=0.999948
  05d676834dbe  AUROC 0.4287->0.4290 (d=+0.0003)   rho 0.1175->0.1169 (d=-0.0006)   corr=0.999826
  0c5ff01c7bfb  AUROC 0.8305->0.8307 (d=+0.0002)   rho 0.6964->0.6967 (d=+0.0004)   corr=0.999989

n compared:        3
worst |d AUROC|:   0.0003
worst |d rho|:     0.0006
min map corr:      0.999826
METRICS STABLE.
  -> Generate the FINE-TUNED arm only (~1h40 for 154 images).
  -> Compare against your existing Stage 9 baseline maps.


,image_id,prompt_finding,auroc_old,auroc_new,d_auroc,rho_old,rho_new,d_rho,map_corr
0,01570ee44031e4ebab6031501293bf66,Consolidation,0.736340,0.736387,0.000048,0.620692,0.620690,-0.000003,0.999948
1,05d676834dbed1639cb5eea70c1e307b,Consolidation,0.428681,0.428994,0.000313,0.117519,0.116913,-0.000605,0.999826
2,0c5ff01c7bfb4362fcd98f36e555b08c,Consolidation,0.830498,0.830691,0.000192,0.696387,0.696743,0.000356,0.999989


In [9]:
import numpy as np
import pandas as pd

# ---------------- CONFIG ----------------
CLASS_MAP_DIRS = {
    "Consolidation": ("/kaggle/input/datasets/kartikichandratre/consolidation-uncertainty-test"),
    "Cardiomegaly":  ("/kaggle/input/datasets/kartikichandratre/cardiomegaly-uncertainty-test"),
}
BASELINE_EDGE_CSV = "/kaggle/input/datasets/kartikichandratre/ext-step0-edge-correlation/edge_correlation_baseline.csv"
N_CARDIO    = 100
N_STRATA    = 10
RANDOM_SEED = 42
# ----------------------------------------

# 1. Stratified Cardiomegaly subsample (stratify on edge_rho, not random)
edge_base = pd.read_csv(BASELINE_EDGE_CSV)
cardio = edge_base[edge_base.class_name == "Cardiomegaly"].copy()
print(f"Cardiomegaly baseline: {len(cardio)} images")

cardio["stratum"] = pd.qcut(cardio.edge_rho, q=N_STRATA,
                            labels=False, duplicates="drop")
per_stratum = max(1, N_CARDIO // cardio.stratum.nunique())
cardio_sample = (cardio.groupby("stratum", group_keys=False)
                       .apply(lambda g: g.sample(min(len(g), per_stratum),
                                                 random_state=RANDOM_SEED)))
cardio_ids = set(cardio_sample.image_id.astype(str))

print(f"  sampled {len(cardio_ids)} across {cardio.stratum.nunique()} strata")
print(f"  edge_rho range  full [{cardio.edge_rho.min():.3f}, "
      f"{cardio.edge_rho.max():.3f}]   "
      f"sample [{cardio_sample.edge_rho.min():.3f}, "
      f"{cardio_sample.edge_rho.max():.3f}]")
print(f"  edge_rho mean   full {cardio.edge_rho.mean():.4f}   "
      f"sample {cardio_sample.edge_rho.mean():.4f}")

span_full = cardio.edge_rho.max() - cardio.edge_rho.min()
span_samp = cardio_sample.edge_rho.max() - cardio_sample.edge_rho.min()
assert span_samp > 0.8 * span_full, \
    f"sample spans only {span_samp:.3f} of {span_full:.3f} -- stratification failed"
print(f"  span retained: {100 * span_samp / span_full:.1f}%  [need >80%]")

# 2. Which image_ids have baseline maps
print("\nBaseline maps available:")
target_ids = set()
for cls, d in CLASS_MAP_DIRS.items():
    ids = set(load_map_dir(d, method="diffusion")[0].keys())
    n_all = len(ids)
    if cls == "Cardiomegaly":
        ids &= cardio_ids
    target_ids |= ids
    print(f"  {cls:15s} {n_all:4d} maps -> targeting {len(ids)}")
print(f"\ntotal targeted: {len(target_ids)}")

# 3. Build and filter the generation frame
tdf_all = build_target_df(["Consolidation", "Cardiomegaly"])

in_test = tdf_all.image_id.astype(str).isin(target_ids).sum()
assert in_test > 0, (
    "ZERO targeted ids found in build_target_df output. METADATA_CSV is "
    "almost certainly still vindr_cxr_train.csv -- point it at "
    "vindr_cxr_test.csv and re-run this cell."
)

tdf = tdf_all[tdf_all.image_id.astype(str).isin(target_ids)].reset_index(drop=True)
print(f"\ngenerating {len(tdf)} images")
print(tdf.finding.value_counts().to_string())

# 4. Final checks
missing = target_ids - set(tdf.image_id.astype(str))
if missing:
    print(f"\n  WARNING: {len(missing)} targeted ids absent from metadata, "
          f"e.g. {sorted(missing)[:3]}")
    print("  These have a baseline map but no fine-tuned pair and will be")
    print("  dropped at the Step 6 merge. Investigate before generating.")

counts = tdf.finding.value_counts()
n_cons = counts.get("Consolidation", 0)
n_card = counts.get("Cardiomegaly", 0)
if n_card < 0.8 * len(cardio_ids):
    print(f"\n  WARNING: only {n_card} Cardiomegaly of {len(cardio_ids)} targeted.")
    print("  first_match() is absorbing multi-label images into Consolidation.")
    print("  Consistent with the ORIGINAL run, but it shifts the per-class n.")

est_min = len(tdf) * 38.6 / 60
print(f"\nestimated generation time: {est_min:.0f} min "
      f"({len(tdf)} imgs x 38.6 s, 2x T4)")
print(f"  Consolidation {n_cons}  |  Cardiomegaly {n_card}")

Cardiomegaly baseline: 331 images
  sampled 100 across 10 strata
  edge_rho range  full [-0.035, 0.748]   sample [-0.035, 0.748]
  edge_rho mean   full 0.4862   sample 0.4860
  span retained: 100.0%  [need >80%]

Baseline maps available:


/tmp/ipykernel_58/41761586.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), per_stratum),


  Consolidation     54 maps -> targeting 54
  Cardiomegaly     331 maps -> targeting 100

total targeted: 150
targeting 374 images across ['Consolidation', 'Cardiomegaly']
finding
Cardiomegaly     331
Consolidation     43
Name: count, dtype: int64

generating 150 images
finding
Cardiomegaly     107
Consolidation     43

estimated generation time: 96 min (150 imgs x 38.6 s, 2x T4)
  Consolidation 43  |  Cardiomegaly 107


In [10]:
import gc, time
from pathlib import Path
import numpy as np
import torch

# ---------------- CONFIG ----------------
LORA_PATH = "/kaggle/input/datasets/kartikichandratre/roentgen-lora-cxr-rank8/lora_cxr"   # from Cell 4
OUT_FT    = Path("/kaggle/working/finetuned_maps")
CHUNK     = 50
# ----------------------------------------

# Preflight -- cheap insurance against a wrong path costing 100 minutes
lp = Path(LORA_PATH)
assert lp.exists(), f"LORA_PATH does not exist: {LORA_PATH}"
weights = list(lp.glob("*.safetensors")) + list(lp.glob("*.bin"))
assert weights, (f"No .safetensors/.bin in {LORA_PATH}. Contents: "
                 f"{[p.name for p in lp.iterdir()][:10]}")
print(f"LoRA weights: {[w.name for w in weights]}")
print(f"generating {len(tdf)} images -> {OUT_FT}")
print(f"estimated {len(tdf) * 38.6 / 60:.0f} min\n")

# Generate. BOTH GPUs must print "LoRA fused" -- if one is silent, half the
# maps would be baseline and the comparison is meaningless.
pipes, scheds, inv_scheds = load_all_pipelines(lora_path=LORA_PATH)
print()

OUT_FT.mkdir(parents=True, exist_ok=True)
chunks = [tdf.iloc[i:i + CHUNK] for i in range(0, len(tdf), CHUNK)]
t0 = time.time()

for i, ch in enumerate(chunks):
    print(f"\n--- chunk {i + 1}/{len(chunks)} ({len(ch)} images) ---")
    res = process_chunk(ch, pipes, scheds, inv_scheds)
    save_chunk_npz(res, i, OUT_FT)
    done = sum(len(c) for c in chunks[:i + 1])
    el = (time.time() - t0) / 60
    print(f"    {done}/{len(tdf)} done | {el:.0f} min elapsed | "
          f"~{el / done * (len(tdf) - done):.0f} min remaining")

print(f"\nfinished in {(time.time() - t0) / 60:.1f} min")

# Verify before releasing the GPUs
written = sorted(OUT_FT.glob("uncertainty_chunk_*.npz"))
print(f"\n{len(written)} chunk file(s):")
total_imgs = 0
for p in written:
    with np.load(p) as z:
        n = len({k.rpartition("__")[0] for k in z.files})
    total_imgs += n
    print(f"  {p.name}  {n} images  {p.stat().st_size / 1e6:.1f} MB")
print(f"\ntotal images written: {total_imgs} / {len(tdf)}")
if total_imgs < len(tdf):
    print(f"  WARNING: {len(tdf) - total_imgs} missing -- check FAILED lines above")

# Degenerate-variance check: fuse_lora() breaking the stochastic path would
# give all-zero maps, which would only surface as nonsense in Step 6.
with np.load(written[0]) as z:
    for k in [k for k in z.files if k.endswith("__variance")][:3]:
        v = z[k]
        flag = "ALL ZERO -- LoRA broke sampling" if np.allclose(v, 0) else "ok"
        print(f"  {k[:12]}... range [{v.min():.3e}, {v.max():.3e}]  {flag}")

del pipes, scheds, inv_scheds
gc.collect(); torch.cuda.empty_cache()

print("\n" + "=" * 62)
print("PUBLISH NOW, BEFORE RUNNING CELLS 9-11.")
print("Output panel -> New Dataset (or a new version).")
print("Cells 9-11 are cheap to re-run from saved maps. This 1h40 is not.")
print("=" * 62)

LoRA weights: ['pytorch_lora_weights.safetensors']
generating 150 images -> /kaggle/working/finetuned_maps
estimated 96 min



Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:0] loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


...

[cuda:1]:   0%|          | 0/25 [00:00<?, ?it/s]

  saved chunk 1: 50 imgs -> uncertainty_chunk_001.npz (89.8 MB)
    100/150 done | 66 min elapsed | ~33 min remaining

--- chunk 3/3 (50 images) ---


[cuda:1]:   0%|          | 0/25 [00:00<?, ?it/s]

[cuda:0]:   0%|          | 0/25 [00:00<?, ?it/s]

  saved chunk 2: 50 imgs -> uncertainty_chunk_002.npz (89.6 MB)
    150/150 done | 99 min elapsed | ~0 min remaining

finished in 99.1 min

3 chunk file(s):
  uncertainty_chunk_000.npz  50 images  89.9 MB
  uncertainty_chunk_001.npz  50 images  89.8 MB
  uncertainty_chunk_002.npz  50 images  89.6 MB

total images written: 150 / 150
  008b3176a724... range [6.367e-05, 1.366e-01]  ok
  010018c93ed3... range [3.698e-07, 7.175e-02]  ok
  011244ab511b... range [1.295e-06, 1.055e-01]  ok

PUBLISH NOW, BEFORE RUNNING CELLS 9-11.
Output panel -> New Dataset (or a new version).
Cells 9-11 are cheap to re-run from saved maps. This 1h40 is not.


In [12]:
import json, os, shutil, subprocess
from pathlib import Path

# ---------------- config ----------------
WORKING       = Path("/kaggle/working")
DATASET_SLUG  = "ext-step7-lora-rank8"      # lowercase, hyphens, 3-50 chars
DATASET_TITLE = "ext-finetuned-con-cardio-gen"   # 6-50 chars
STAGING       = Path("/kaggle/_ds_staging")    # OUTSIDE /kaggle/working on purpose
# ----------------------------------------

# 1. credentials from Secrets
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
username = sec.get_secret("KAGGLE_USERNAME")
key      = sec.get_secret("KAGGLE_KEY")

kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
(kdir / "kaggle.json").write_text(json.dumps({"username": username, "key": key}))
(kdir / "kaggle.json").chmod(0o600)
os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = username, key
print(f"credentials set for {username}")

# 2. copy EVERYTHING from /kaggle/working
if STAGING.exists():
    shutil.rmtree(STAGING)
shutil.copytree(WORKING, STAGING)

# drop junk that shouldn't ship
for pattern in ["**/__pycache__", "**/.ipynb_checkpoints", "**/.git"]:
    for p in STAGING.glob(pattern):
        shutil.rmtree(p, ignore_errors=True)

total = 0
print(f"\nstaged from {WORKING}:")
for p in sorted(STAGING.rglob("*")):
    if p.is_file():
        mb = p.stat().st_size / 1e6
        total += mb
        print(f"  {p.relative_to(STAGING)}  {mb:.2f} MB")
print(f"\ntotal: {total:.1f} MB  ({sum(1 for p in STAGING.rglob('*') if p.is_file())} files)")
assert total > 0, "/kaggle/working is empty"

# 3. metadata
meta = {"title": DATASET_TITLE,
        "id": f"{username}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}]}
(STAGING / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
print(f"id: {meta['id']}")

# 4. create, or version if it already exists
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    return r.returncode

if run(f'kaggle datasets create -p "{STAGING}" --dir-mode zip') != 0:
    print("create failed — trying as a new version")
    run(f'kaggle datasets version -p "{STAGING}" -m "step3 lora + manifest" --dir-mode zip')

print(f"\nhttps://www.kaggle.com/datasets/{username}/{DATASET_SLUG}")

credentials set for kartikichandratre

staged from /kaggle/working:
  .virtual_documents/__notebook_source__.ipynb  0.07 MB
  finetuned_maps/uncertainty_chunk_000.npz  89.87 MB
  finetuned_maps/uncertainty_chunk_000_meta.csv  0.00 MB
  finetuned_maps/uncertainty_chunk_001.npz  89.81 MB
  finetuned_maps/uncertainty_chunk_001_meta.csv  0.00 MB
  finetuned_maps/uncertainty_chunk_002.npz  89.61 MB
  finetuned_maps/uncertainty_chunk_002_meta.csv  0.00 MB

total: 269.4 MB  (7 files)
id: kartikichandratre/ext-step7-lora-rank8
Starting upload for file .virtual_documents.zip
Upload successful: .virtual_documents.zip (22KB)
Starting upload for file finetuned_maps.zip
Upload successful: finetuned_maps.zip (257MB)
Dataset creation error: The requested title "ext-finetuned-con-cardio-gen" is already in use by a dataset. Please choose another title.
 
  0%|          | 0.00/22.4k [00:00<?, ?B/s]
100%|██████████| 22.4k/22.4k [00:00<00:00, 58.4kB/s]

  0%|          | 0.00/257M [00:00<?, ?B/s]
  4%|▎   

In [1]:
TEST_MAP_DIRS = {
    "Consolidation": {"diffusion": str(/kaggle/input/datasets/kartikichandratre/consolidation-uncertainty-test), "mcd_logit": None, "mcd_gradcam": None},
    "Cardiomegaly":  {"diffusion": str(OUT_FT), "mcd_logit": None, "mcd_gradcam": None},
}
OUTPUT_DIR = "/kaggle/working/stage9_ft"
per_image_ft, summary_ft = main()

NameError: name 'OUT_FT' is not defined

In [5]:
"""
step0_edge_correlation_baseline.py
============================================================================
RUN THIS FIRST. No GPU. ~10 minutes.

Measures the edge-variance correlation on your EXISTING baseline maps,
before any fine-tuning happens. This banks a publishable metric
immediately, so that even if LoRA training fails or the session dies,
you still have a new quantitative result to put on the CV.

WHAT IT MEASURES
Per-image Spearman correlation between a Sobel edge map of the radiograph
and the diffusion variance map. The thesis claims variance is
edge-dominated (peak variance inside a lesion box in 0 of 547 images).
This turns that qualitative claim into a number.

Validated behaviour on synthetic data:
    variance == edge map        rho = 1.00
    variance == pure noise      rho = 0.00
    edge-dominated + lesion     rho = 0.95
    lesion-dominated            rho = 0.38
So a high rho means variance tracks anatomy, not pathology.

PASTE AS A CELL AFTER stage9_final.py (needs its loaders + config).

OUTPUT: edge_correlation_baseline.csv, plus a printed per-class summary
that is the "before" column of your fine-tuning scorecard.
"""
DRIVER_MODE = True
import numpy as np
import pandas as pd
import h5py
from scipy.ndimage import sobel
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# Reuse the H5 config from plot_variance_heatmap_examples.py
H5_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5"

def h5_get_image(h5file, image_id):
    return h5file[image_id][()]

OUTPUT_CSV = "/kaggle/working/edge_correlation_baseline.csv"


def edge_variance_correlation(img, var_map):
    """Spearman rho between Sobel edge magnitude and the variance map."""
    img = img.astype(np.float64)
    edges = np.hypot(sobel(img, axis=0), sobel(img, axis=1))
    rho, p = spearmanr(edges.ravel(), var_map.ravel())
    return float(rho), float(p)


def main():
    rows = []
    with h5py.File(H5_PATH, "r") as h5file:
        for class_name, paths in TEST_MAP_DIRS.items():
            d = paths.get("diffusion")
            if d is None:
                continue
            maps, _ = load_map_dir(d, method="diffusion")
            print(f"{class_name}: {len(maps)} maps")
            for image_id, var_map in tqdm(maps.items(), desc=class_name, leave=False):
                try:
                    img = h5_get_image(h5file, image_id)
                except KeyError:
                    print(f"  WARNING: {image_id} not in H5, skipping")
                    continue
                rho, p = edge_variance_correlation(img, var_map)
                rows.append({"class_name": class_name, "image_id": image_id,
                             "edge_rho": rho, "p": p})

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)

    print("\n" + "=" * 62)
    print("BASELINE EDGE-VARIANCE CORRELATION  (zero-shot generator)")
    print("=" * 62)
    summary = df.groupby("class_name")["edge_rho"].agg(
        ["count", "mean", "std", "min", "max"])
    print(summary.round(4).to_string())
    print(f"\nPooled across all classes: mean rho = {df['edge_rho'].mean():.4f} "
          f"(n = {len(df)})")
    print("\nThis is the 'before' column of the fine-tuning scorecard.")
    print("A DROP after fine-tuning = domain adaptation moved variance away")
    print("from anatomical boundaries. No drop = edge-dominance is intrinsic.")
    return df



main()

Pneumothorax: 18 maps


Pneumothorax:   0%|          | 0/18 [00:00<?, ?it/s]

Consolidation: 54 maps


Consolidation:   0%|          | 0/54 [00:00<?, ?it/s]

Nodule/Mass: 117 maps


...

Cardiomegaly:   0%|          | 0/331 [00:00<?, ?it/s]

Atelectasis: 27 maps


Atelectasis:   0%|          | 0/27 [00:00<?, ?it/s]


BASELINE EDGE-VARIANCE CORRELATION  (zero-shot generator)
               count    mean     std     min     max
class_name                                          
Atelectasis       27  0.3861  0.1632  0.0126  0.6418
Cardiomegaly     331  0.4862  0.1529 -0.0354  0.7484
Consolidation     54  0.4632  0.1506  0.1164  0.7508
Nodule/Mass      117  0.4335  0.1672  0.0170  0.7102
Pneumothorax      18  0.4447  0.1580  0.1239  0.6415

Pooled across all classes: mean rho = 0.4663 (n = 547)

This is the 'before' column of the fine-tuning scorecard.
A DROP after fine-tuning = domain adaptation moved variance away
from anatomical boundaries. No drop = edge-dominance is intrinsic.


,class_name,image_id,edge_rho,p
0,Pneumothorax,05d676834dbed1639cb5eea70c1e307b,0.127082,0.000000e+00
1,Pneumothorax,2f5a3aa315379bb01b8b4c9a1ece8e2e,0.600844,0.000000e+00
2,Pneumothorax,3479c81736f275a848b74d952ebfab29,0.485473,0.000000e+00
3,Pneumothorax,3c63e58fcda26e02fdd6619515399985,0.475471,0.000000e+00
4,Pneumothorax,4308b795084095f21117491e3b07f2a7,0.498150,0.000000e+00
...,...,...,...,...
542,Atelectasis,4d911f55a3576833aa411f5718c8021e,0.630672,0.000000e+00
543,Atelectasis,508083b00dfef2ea10fe2aebee580990,0.539322,0.000000e+00
544,Atelectasis,5ff1e6bd7cb14179a70db035d3aa6ba6,0.012576,1.202439e-10
545,Atelectasis,66c570971c7df4782e3dfb9e74f0dd1e,0.376825,0.000000e+00
